# Hurst Regimes 03 — volatility roughness with seasonality, mean reversion and SMM

**Discussion:** `discussions/Hurst_Regimes/005_claude_03_review_fixes.md` · **Code:** `src/data_engineering/modeling/rough_vol_smm.py`

This notebook implements the HR 004 review fixes.

1. **Seasonality.** Returns are deseasonalized by day-of-week × time-of-day. Lags at multiples of 24 h and 7 days are excluded from every fit.
2. **Mean reversion as its own parameter.** Discrete fOU + noise structure function on hourly $\log\sigma$:
   $$m_2(k)=2V\,[1-\rho_{fOU}(k;H,\kappa)]+2\eta^2.$$
   This treats hourly log RV as a point value (integration over the hour is ignored), so it is an approximation.
3. **Primary estimator: simulated method of moments over $(H,\kappa,\nu,\text{df})$.**
   - Model: $\log\sigma$ follows an fOU at 5-minute steps; returns are $s_{dow,tod}\,\sigma_t\,\varepsilon_t$ with $\varepsilon\sim t(\text{df})$.
   - Each simulation runs on the data's own timestamps, gaps and seasonal profile, and goes through the identical pipeline (seasonal re-estimation, 1h/4h/1d RV, logs, 18 moments).
   - Common random numbers, 10 replicates, $H$ grid from 0.005, sandwich standard errors.
   - Parameter recovery is checked on synthetic data with the same layout before the real fits.

Descriptive only: no gate, no trading claim.

Environment (optional): `HR_HISTORY_DIR`, `HR_ASSETS`, `HR03_RECOVERY` (default 1; set 0 to skip the about 20-minute recovery check).

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_engineering.modeling import hurst_regimes as hr
from data_engineering.modeling import hurst_roughness as rg
from data_engineering.modeling import rough_vol_smm as sm

HIST_DIR = Path(os.getenv('HR_HISTORY_DIR', str(DATA_ROOT / 'history' / 'price_5m')))
ASSETS = [x.strip().upper() for x in os.getenv('HR_ASSETS', 'AVAX,BTC,ETH,SOL,TRX,XRP').split(',') if x.strip()]
RECOVERY = os.getenv('HR03_RECOVERY', '1').strip() != '0'
OUT = PROJECT_ROOT / 'notebooks' / 'Hurst_Regimes' / '03'
OUT.mkdir(parents=True, exist_ok=True)
print('Hurst Regimes 03', sm.VERSION, sm.selftest())
hist = hr.load_history(HIST_DIR, ASSETS)
DATA = {}
for a in sorted(hist.underlying.unique()):
    lp = rg.price_grid(hist, a)
    if lp.notna().sum() < 288 * rg.MIN_DAYS:
        print(f'{a}: excluded (short history)'); continue
    r, slot, idx = sm.day_grid(lp)
    DATA[a] = dict(r=r, slot=slot, idx=idx)
    print(f'{a}: {idx[0]} -> {idx[-1]} | days {len(r) // 288} | missing {np.mean(~np.isfinite(r)):.2%}')

## 1. Seasonality: day-of-week × time-of-day (fix 1)

In [ ]:
season_rows, desc_rows = [], []
fig, ax = plt.subplots(figsize=(11, 3))
for a, d in DATA.items():
    prof = sm.dow_tod_profile(d['r'], d['slot'])
    d['prof'] = prof
    lv_dt = sm.log_vol_all(d['r'], d['slot'], prof)
    lv_t = sm.log_vol_all(d['r'], d['slot'], sm.tod_profile(d['r'], d['slot']))
    d['lv'] = lv_dt
    a1 = rg.acf(lv_t['1h'], 170); a2 = rg.acf(lv_dt['1h'], 170)
    dow_level = (prof.reshape(7, 288) ** 2).mean(1) ** 0.5
    season_rows.append({'underlying': a, 'acf1h_lag24_tod': a1[24], 'acf1h_lag24_dowtod': a2[24],
                        'acf1h_lag168_tod': a1[168], 'acf1h_lag168_dowtod': a2[168], 'acf1h_lag100_dowtod': a2[100],
                        **{f'rms_{dn}': v for dn, v in zip(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], dow_level)}})
    ax.plot(np.arange(2016) / 288, prof, lw=0.6, label=a)
    for b, lags in sm.DESC_LAGS.items():
        sf = rg.structure_function(lv_dt[b], lags)
        z = rg.zeta(sf, 1, max(lags)); nc = rg.noise_corrected_H(sf, 1, max(lags))
        desc_rows.append({'underlying': a, 'block': b, 'lags': str(lags), 'H_raw': z.get('H2'), 'H_noise_corr': nc.get('H_noise_corr'),
                          'noise_share_lag1': nc.get('noise_share_lag1')})
ax.set_xlabel('day of week (0 = Monday)'); ax.set_title('dow x tod RMS profile'); ax.legend(ncol=6); plt.show()
season_tab, desc_tab = pd.DataFrame(season_rows), pd.DataFrame(desc_rows)
display(season_tab.round(3))
print('Descriptive H on pre-declared short lag ranges (no 24h/168h multiples), dow x tod deseasonalized:')
display(desc_tab.pivot_table(index='underlying', columns='block', values=['H_raw', 'H_noise_corr']).round(3))

## 2. Parametric fOU + noise structure function on hourly $\log\sigma$ (fix 2, point approximation)

In [ ]:
param_rows = []
t0 = time.time()
for a, d in DATA.items():
    param_rows.append({'underlying': a, **sm.param_fit_1h(d['lv']['1h'])})
    print(f'{a}: done ({time.time() - t0:.0f}s)')
param_tab = pd.DataFrame(param_rows)
display(param_tab.round(4))

## 3. SMM recovery check on synthetic data with the real layout (fix 3)

In [ ]:
rec_rows = []
if RECOVERY:
    a0 = list(DATA)[0]
    d = DATA[a0]
    mask = ~np.isfinite(d['r'])
    for th in [(0.05, 0.05, 0.4, 5.0), (0.02, 0.2, 0.4, 4.0)]:
        t0 = time.time()
        rec_rows.append({'layout': a0, **sm.recovery_check(th, d['slot'], mask, d['prof'])})
        print(f'recovery {th}: {time.time() - t0:.0f}s')
rec_tab = pd.DataFrame(rec_rows)
display(rec_tab.round(4))

## 4. SMM estimates of $(H,\kappa,\nu,\text{df})$

In [ ]:
smm_rows, mom_tabs = [], []
for a, d in DATA.items():
    t0 = time.time()
    f = sm.fit(d['r'], d['slot'])
    smm_rows.append({'underlying': a, **f['summary']})
    mom_tabs.append(f['moments'].assign(underlying=a))
    print(f"{a}: H={f['summary']['H']:.3f} [{f['summary']['H_ci_low']:.3f}, {f['summary']['H_ci_high']:.3f}] "
          f"kappa/day={f['summary']['kappa_per_day']:.4f} half-life={f['summary']['half_life_days']:.1f} d "
          f"df={f['summary']['df']:.1f} max|t|={f['summary']['max_abs_moment_t']:.2f} ({time.time() - t0:.0f}s)")
smm_tab = pd.DataFrame(smm_rows)
mom_tab = pd.concat(mom_tabs, ignore_index=True)
display(smm_tab.round(4))
display(mom_tab.pivot_table(index='moment', columns='underlying', values='t').reindex(sm.MOMENT_NAMES).round(2))

## 5. Outputs

In [ ]:
season_tab.to_csv(OUT / '03_seasonality.csv', index=False, float_format='%.5g')
desc_tab.to_csv(OUT / '03_descriptive_H.csv', index=False, float_format='%.5g')
param_tab.to_csv(OUT / '03_param_fou_1h.csv', index=False, float_format='%.6g')
rec_tab.to_csv(OUT / '03_smm_recovery.csv', index=False, float_format='%.5g')
smm_tab.to_csv(OUT / '03_smm_estimates.csv', index=False, float_format='%.6g')
mom_tab.to_csv(OUT / '03_smm_moment_fit.csv', index=False, float_format='%.5g')
meta = {'module': '03_rough_vol_smm', 'version': sm.VERSION, 'assets': list(DATA), 'recovery_run': RECOVERY,
        'r_final': sm.R_FINAL, 'r_cov': sm.R_COV, 'h_grid': list(sm.H_GRID), 'kappa_grid_per_day': list(sm.KAPPA_GRID),
        'moments': sm.MOMENT_NAMES, 'history_dir': str(HIST_DIR)}
(OUT / '03_run_metadata.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))